In [ ]:
import os
import torch
import xml.etree.ElementTree as ET
from PIL import Image
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

# Base faster rcnn model training

# Define SOHAS dataset and the image and annotation directories
class SOHASDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        # Names of the classes in the SOHAS dataset
        self.all_classes = ["knife", "pistol", "smartphone", "billete", "monedero", "tarjeta"]
        self.class_to_idx = {k: v + 1 for v, k in enumerate(self.all_classes)}

        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

      # Finding the training images and the groundtruths for the objects
    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".xml"))

        img = Image.open(img_path).convert("RGB")
        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.all_classes:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[label])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        if self.transforms:
            img = self.transforms(img)
        else:
            img = ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# Load the faster rcnn model and cutomise it for the dataset
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# Training loop for the model
def train_one_epoch(model, dataloader, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(dataloader, desc="Training"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        with autocast():
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    return running_loss / len(dataloader)

#Filter for the knife and pistol classes
def evaluate(model, dataloader, device, idx_to_class):
    model.eval()
    with torch.no_grad():
        for i, (images, _) in enumerate(tqdm(dataloader, desc="Evaluating")):
            images = [img.to(device) for img in images]
            outputs = model(images)
            for j, output in enumerate(outputs):
                print(f"\nImage {i * len(images) + j}:")
                for box, label, score in zip(output["boxes"], output["labels"], output["scores"]):
                    class_name = idx_to_class[label.item()]
                    if class_name in ["knife", "pistol"] and score > 0.5:
                        print(f"Detected {class_name} with {score:.2f} confidence at {box.tolist()}")

# Main function for the training and testing of the model
def main():
    # Assign the pathways to the train and test datasets
    train_img_dir = "/content/drive/MyDrive/SOHAS/train_dataset/images"
    train_ann_dir = "/content/drive/MyDrive/SOHAS/train_dataset/annotations"
    test_img_dir = "/content/drive/MyDrive/SOHAS/test_dataset/images"
    test_ann_dir = "/content/drive/MyDrive/SOHAS/test_dataset/annotations"

    # GPU setup to use the collab GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)


    transform = ToTensor()
    train_dataset = SOHASDataset(train_img_dir, train_ann_dir, transform)
    test_dataset = SOHASDataset(test_img_dir, test_ann_dir, transform)

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                              num_workers=4, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False,
                             num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))

    # Set the classes for the training
    num_classes = 1 + len(train_dataset.all_classes)
    idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}


    model = get_model(num_classes).to(device)

    # Optimiser
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
    scaler = GradScaler()

    # Start the training of the model
    num_epochs = 10
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        avg_loss = train_one_epoch(model, train_loader, optimizer, device, scaler)
        print(f"Average Loss: {avg_loss:.4f}")

        # Save model after each epoch
        torch.save(model.state_dict(), f"fasterrcnn_epoch{epoch+1}.pth")

    # Evaluate the trained model
    evaluate(model, test_loader, device, idx_to_class)

if __name__ == "__main__":
    main()



Using device: cuda


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:00<00:00, 172MB/s]
<ipython-input-1-8025b6ae17f3>:143: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Epoch 1/10


Training:   0%|          | 0/1208 [00:00<?, ?it/s]<ipython-input-1-8025b6ae17f3>:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # Mixed precision context
Training: 100%|██████████| 1208/1208 [15:12<00:00,  1.32it/s]


Average Loss: 0.1621

Epoch 2/10


...


Image 822:

Image 823:


Evaluating: 100%|█████████▉| 413/414 [04:52<00:00,  1.19it/s]


Image 824:

Image 825:


Evaluating: 100%|██████████| 414/414 [04:52<00:00,  1.41it/s]


Image 826:

Image 827:


In [ ]:
import os
import torch
import torchmetrics
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from tqdm import tqdm
import numpy as np
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report

# Define the SOHAS dataset
class SOHASDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        # Names of the object classes for the ground truths
        self.all_classes = ["knife", "pistol", "smartphone", "billete", "monedero", "tarjeta"]
        self.class_to_idx = {k: v + 1 for v, k in enumerate(self.all_classes)}

        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".xml"))

        img = Image.open(img_path).convert("RGB")
        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.all_classes:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[label])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        if self.transforms:
            img = self.transforms(img)
        else:
            img = ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# Load the faster rcnn model
def get_model(num_classes, checkpoint_path=None, device='cuda'):
    # Load a pre-trained faster rcnn model
    model = fasterrcnn_resnet50_fpn(pretrained=True)


    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)


    if checkpoint_path:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    model.to(device)
    return model

# Test and evaluate the model for weapon vs non weapon
def evaluate(model, dataloader, device, idx_to_class):

    y_true_weapon = []
    y_pred_weapon = []


    y_true_knife_pistol = []
    y_pred_knife_pistol = []

    # Accuracy checker
    correct_weapon = 0
    total_weapon = 0
    correct_knife_pistol = 0
    total_knife_pistol = 0

    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Run the model
            outputs = model(images)

            # Weapon vs non weapon classification
            for target, output in zip(targets, outputs):
                labels = target["labels"]
                pred_labels = output["labels"]
                for true, pred in zip(labels, pred_labels):

                    true_weapon = 1 if idx_to_class[true.item()] in ['knife', 'pistol'] else 0
                    pred_weapon = 1 if idx_to_class[pred.item()] in ['knife', 'pistol'] else 0

                    y_true_weapon.append(true_weapon)
                    y_pred_weapon.append(pred_weapon)


                    if true_weapon == 1:
                        true_knife_pistol = 1 if idx_to_class[true.item()] == 'knife' else 2
                        pred_knife_pistol = 1 if idx_to_class[pred.item()] == 'knife' else 2
                        y_true_knife_pistol.append(true_knife_pistol)
                        y_pred_knife_pistol.append(pred_knife_pistol)


                    if true_weapon == pred_weapon:
                        correct_weapon += 1
                    total_weapon += 1

                    if true_knife_pistol == pred_knife_pistol:
                        correct_knife_pistol += 1
                    total_knife_pistol += 1

    # Evaluate the weapon detection
    precision_weapon = precision_score(y_true_weapon, y_pred_weapon)
    recall_weapon = recall_score(y_true_weapon, y_pred_weapon)
    f1_weapon = f1_score(y_true_weapon, y_pred_weapon)
    accuracy_weapon = accuracy_score(y_true_weapon, y_pred_weapon)

    # Evaluate the weapon identification
    precision_knife_pistol = precision_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    recall_knife_pistol = recall_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    f1_knife_pistol = f1_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    accuracy_knife_pistol = accuracy_score(y_true_knife_pistol, y_pred_knife_pistol)

    # Print the evaluation results for the the detection and identification
    print("\n[Detection] Weapon vs Non-Weapon:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'No Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {len(y_true_weapon) - sum(y_true_weapon)}")
    print(f"{'Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {sum(y_true_weapon)}")
    print(f"\n    accuracy                           {accuracy_weapon:.2f}       {len(y_true_weapon)}")
    print(f"   macro avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")
    print(f"weighted avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")

    print("\n[Classification] Knife vs Pistol:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'weapon':<20} {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}      {len(y_true_knife_pistol)}")
    print(f"\n    accuracy                           {accuracy_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"   macro avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"weighted avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")

def main():
    # Paths to the test dataset and the trained faster rcnn model
    test_img_dir = "/content/drive/MyDrive/SOHAS/test_dataset/images"
    test_ann_dir = "/content/drive/MyDrive/SOHAS/test_dataset/annotations"
    pretrained_model_path = "/content/fasterrcnn_epoch10.pth"

    # GPU setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)


    test_dataset = SOHASDataset(test_img_dir, test_ann_dir, transforms=ToTensor())

    print(f"Number of images in the test dataset: {len(test_dataset)}")

    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))


    idx_to_class = {v: k for k, v in test_dataset.class_to_idx.items()}

    # Load the trained model
    model = get_model(num_classes=1 + len(test_dataset.all_classes), checkpoint_path=pretrained_model_path, device=device)


    evaluate(model, test_loader, device, idx_to_class)

if __name__ == "__main__":
    main()



Using device: cuda
Number of images in the test dataset: 828


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Evaluating: 100%|██████████| 414/414 [01:52<00:00,  3.69it/s]


[Detection] Weapon vs Non-Weapon:
                     precision    recall  f1-score   support
No Weapon            0.93      0.99      0.96      321
Weapon               0.93      0.99      0.96      548

    accuracy                           0.94       869
   macro avg       0.93      0.99      0.96       869
weighted avg       0.93      0.99      0.96       869

[Classification] Knife vs Pistol:
                     precision    recall  f1-score   support
weapon               0.97      0.95      0.96      548

    accuracy                           0.98       548
   macro avg       0.97      0.95      0.96       548
weighted avg       0.97      0.95      0.96       548


In [2]:
pip install torch torchvision torchmetrics scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 62.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Unins

In [ ]:
import os
import torch
import xml.etree.ElementTree as ET
from PIL import Image
import cv2
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import ToTensor
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

torch.backends.cudnn.benchmark = True

# Augmentation for the train images
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
        A.Resize(640, 640),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

def get_test_transforms():
    return A.Compose([
        A.Resize(640, 640),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# SOHAS dataset logic
class SOHASDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        self.all_classes = ["knife", "pistol", "smartphone", "billete", "monedero", "tarjeta"]
        self.class_to_idx = {k: v + 1 for v, k in enumerate(self.all_classes)}
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".xml"))

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.all_classes:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[label])

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            transformed = self.transforms(image=img, bboxes=target["boxes"], class_labels=target["labels"])
            img = transformed['image']
            boxes = torch.tensor(transformed['bboxes'], dtype=torch.float32)
            labels = torch.tensor(transformed['class_labels'], dtype=torch.int64)
        else:
            img = ToTensor()(Image.fromarray(img))
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)


        if boxes.ndim == 1 or boxes.numel() == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd
        }

        return img, target

    def __len__(self):
        return len(self.imgs)

# Load faster rcnn
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# Training loop for the model
def train_one_epoch(model, dataloader, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(dataloader, desc="Training"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    return running_loss / len(dataloader)

# Evaluation of the model
def evaluate(model, dataloader, device, idx_to_class):
    model.eval()
    with torch.no_grad():
        for i, (images, _) in enumerate(tqdm(dataloader, desc="Evaluating")):
            images = [img.to(device) for img in images]
            outputs = model(images)
            for j, output in enumerate(outputs):
                print(f"\nImage {i * len(images) + j}:")
                for box, label, score in zip(output["boxes"], output["labels"], output["scores"]):
                    class_name = idx_to_class[label.item()]
                    if class_name in ["knife", "pistol"] and score > 0.5:
                        print(f"✅ Detected {class_name.upper()} with {score:.2f} confidence at {box.tolist()}")

#Paths to the training and testing datasets
def main():
    train_img_dir = "/content/drive/MyDrive/SOHAS/train_dataset/images"
    train_ann_dir = "/content/drive/MyDrive/SOHAS/train_dataset/annotations"
    test_img_dir = "/content/drive/MyDrive/SOHAS/test_dataset/images"
    test_ann_dir = "/content/drive/MyDrive/SOHAS/test_dataset/annotations"

    #GPU setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    train_dataset = SOHASDataset(train_img_dir, train_ann_dir, transforms=get_train_transforms())
    test_dataset = SOHASDataset(test_img_dir, test_ann_dir, transforms=get_test_transforms())

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                              num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False,
                             num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))

    num_classes = 1 + len(train_dataset.all_classes)
    idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}

    model = get_model(num_classes).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
    scaler = GradScaler(device='cuda')


    num_epochs = 20
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        avg_loss = train_one_epoch(model, train_loader, optimizer, device, scaler)
        print(f"Average Loss: {avg_loss:.4f}")
        torch.save(model.state_dict(), f"fasterrcnn_epoch{epoch+1}.pth")

    evaluate(model, test_loader, device, idx_to_class)

if __name__ == "__main__":
    main()



In [7]:
import os
import torch
import torchmetrics
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from tqdm import tqdm
import numpy as np
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report
from PIL import Image

# Define the SOHAS Dataset
class SOHASDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        # Ground truth class names
        self.all_classes = ["knife", "pistol", "smartphone", "billete", "monedero", "tarjeta"]
        self.class_to_idx = {k: v + 1 for v, k in enumerate(self.all_classes)}

        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".xml"))

        img = Image.open(img_path).convert("RGB")
        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.all_classes:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[label])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        if self.transforms:
            img = self.transforms(img)
        else:
            img = ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# Load and customise the faster rcnn model
def get_model(num_classes, checkpoint_path=None, device='cuda'):
    # Load a pre-trained faster rcnn model
    model = fasterrcnn_resnet50_fpn(pretrained=True)


    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)


    if checkpoint_path:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    model.to(device)
    return model

# Evaluate the model for detection and classification
def evaluate(model, dataloader, device, idx_to_class):
    # Metrics for weapon vs non weapon
    y_true_weapon = []
    y_pred_weapon = []

    # Metrics for weapon identification
    y_true_knife_pistol = []
    y_pred_knife_pistol = []

    # Count for Accuracy
    correct_weapon = 0
    total_weapon = 0
    correct_knife_pistol = 0
    total_knife_pistol = 0

    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Run the model on the batch of images
            outputs = model(images)

            # Calculate the classification
            for target, output in zip(targets, outputs):
                labels = target["labels"]
                pred_labels = output["labels"]
                for true, pred in zip(labels, pred_labels):
                   # Weapon detection
                    true_weapon = 1 if idx_to_class[true.item()] in ['knife', 'pistol'] else 0
                    pred_weapon = 1 if idx_to_class[pred.item()] in ['knife', 'pistol'] else 0

                    y_true_weapon.append(true_weapon)
                    y_pred_weapon.append(pred_weapon)

                    # Weapon classification
                    if true_weapon == 1:
                        true_knife_pistol = 1 if idx_to_class[true.item()] == 'knife' else 2
                        pred_knife_pistol = 1 if idx_to_class[pred.item()] == 'knife' else 2
                        y_true_knife_pistol.append(true_knife_pistol)
                        y_pred_knife_pistol.append(pred_knife_pistol)


                    if true_weapon == pred_weapon:
                        correct_weapon += 1
                    total_weapon += 1

                    if true_knife_pistol == pred_knife_pistol:
                        correct_knife_pistol += 1
                    total_knife_pistol += 1

    # Weapon vs Non-Weapon Metrics
    precision_weapon = precision_score(y_true_weapon, y_pred_weapon)
    recall_weapon = recall_score(y_true_weapon, y_pred_weapon)
    f1_weapon = f1_score(y_true_weapon, y_pred_weapon)
    accuracy_weapon = accuracy_score(y_true_weapon, y_pred_weapon)

    # Knife vs Pistol Metrics
    precision_knife_pistol = precision_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    recall_knife_pistol = recall_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    f1_knife_pistol = f1_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    accuracy_knife_pistol = accuracy_score(y_true_knife_pistol, y_pred_knife_pistol)

    # Print results
    print("\n[Detection] Weapon vs Non-Weapon:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'No Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {len(y_true_weapon) - sum(y_true_weapon)}")
    print(f"{'Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {sum(y_true_weapon)}")
    print(f"\n    accuracy                           {accuracy_weapon:.2f}       {len(y_true_weapon)}")
    print(f"   macro avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")
    print(f"weighted avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")

    print("\n[Classification] Knife vs Pistol:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'weapon':<20} {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}      {len(y_true_knife_pistol)}")
    print(f"\n    accuracy                           {accuracy_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"   macro avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"weighted avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")

# Main evaluation pipeline
def main():
    # Paths to the dataset and the pretrained model
    test_img_dir = "/content/drive/MyDrive/SOHAS/test_dataset/images"
    test_ann_dir = "/content/drive/MyDrive/SOHAS/test_dataset/annotations"
    pretrained_model_path = "/content/drive/MyDrive/faster_rcnn_augmentation/fasterrcnn_epoch12.pth"

    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Dataset and dataloader
    test_dataset = SOHASDataset(test_img_dir, test_ann_dir, transforms=ToTensor())

    print(f"Number of images in the test dataset: {len(test_dataset)}")

    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))


    idx_to_class = {v: k for k, v in test_dataset.class_to_idx.items()}

    # Load the trained model
    model = get_model(num_classes=1 + len(test_dataset.all_classes), checkpoint_path=pretrained_model_path, device=device)

    # Evaluate
    evaluate(model, test_loader, device, idx_to_class)

if __name__ == "__main__":
    main()


Using device: cuda
Number of images in the test dataset: 828


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Evaluating: 100%|██████████| 414/414 [00:21<00:00, 19.33it/s]


[Detection] Weapon vs Non-Weapon:
                     precision    recall  f1-score   support
No Weapon            0.90      0.94      0.92      307
Weapon               0.90      0.94      0.92      514

    accuracy                           0.90       821
   macro avg       0.90      0.94      0.92       821
weighted avg       0.90      0.94      0.92       821

[Classification] Knife vs Pistol:
                     precision    recall  f1-score   support
weapon               0.84      0.95      0.88      514

    accuracy                           0.93       514
   macro avg       0.84      0.95      0.88       514
weighted avg       0.84      0.95      0.88       514


In [ ]:
import os
import torch
import xml.etree.ElementTree as ET
from PIL import Image
import cv2
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import ToTensor
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Augmented faster rcnn model img size 825

torch.backends.cudnn.benchmark = True

# Agmentations for the trianing data
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
        A.Resize(825, 825),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

def get_test_transforms():
    return A.Compose([
        A.Resize(640, 640),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# SOHAS dataset directories and the annotation class names
class SOHASDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        self.all_classes = ["knife", "pistol", "smartphone", "billete", "monedero", "tarjeta"]
        self.class_to_idx = {k: v + 1 for v, k in enumerate(self.all_classes)}
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".xml"))

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.all_classes:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[label])

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            transformed = self.transforms(image=img, bboxes=target["boxes"], class_labels=target["labels"])
            img = transformed['image']
            boxes = torch.tensor(transformed['bboxes'], dtype=torch.float32)
            labels = torch.tensor(transformed['class_labels'], dtype=torch.int64)
        else:
            img = ToTensor()(Image.fromarray(img))
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)


        if boxes.ndim == 1 or boxes.numel() == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd
        }

        return img, target

    def __len__(self):
        return len(self.imgs)

# Load faster rcnn
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# Training loop for the model
def train_one_epoch(model, dataloader, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(dataloader, desc="Training"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    return running_loss / len(dataloader)

# Evaluation
def evaluate(model, dataloader, device, idx_to_class):
    model.eval()
    with torch.no_grad():
        for i, (images, _) in enumerate(tqdm(dataloader, desc="Evaluating")):
            images = [img.to(device) for img in images]
            outputs = model(images)
            for j, output in enumerate(outputs):
                print(f"\nImage {i * len(images) + j}:")
                for box, label, score in zip(output["boxes"], output["labels"], output["scores"]):
                    class_name = idx_to_class[label.item()]
                    if class_name in ["knife", "pistol"] and score > 0.5:
                        print(f" Detected {class_name.upper()} with {score:.2f} confidence at {box.tolist()}")

# Pathways to the train/test datasets
def main():
    train_img_dir = "/content/drive/MyDrive/SOHAS/train_dataset/images"
    train_ann_dir = "/content/drive/MyDrive/SOHAS/train_dataset/annotations"
    test_img_dir = "/content/drive/MyDrive/SOHAS/test_dataset/images"
    test_ann_dir = "/content/drive/MyDrive/SOHAS/test_dataset/annotations"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    train_dataset = SOHASDataset(train_img_dir, train_ann_dir, transforms=get_train_transforms())
    test_dataset = SOHASDataset(test_img_dir, test_ann_dir, transforms=get_test_transforms())

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                              num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False,
                             num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))

    num_classes = 1 + len(train_dataset.all_classes)
    idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}

    model = get_model(num_classes).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
    scaler = GradScaler(device='cuda')

    num_epochs = 10
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        avg_loss = train_one_epoch(model, train_loader, optimizer, device, scaler)
        print(f"Average Loss: {avg_loss:.4f}")
        torch.save(model.state_dict(), f"fasterrcnn_epoch{epoch+1}.pth")

    evaluate(model, test_loader, device, idx_to_class)

if __name__ == "__main__":
    main()


Using device: cuda


/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:00<00:00, 215MB/s]



Epoch 1/10


Training: 100%|██████████| 1208/1208 [16:33<00:00,  1.22it/s]


📉 Average Loss: 0.1898

Epoch 2/10


...


Image 816:

Image 817:

Image 818:

Image 819:


Evaluating:  99%|█████████▉| 411/414 [05:03<00:01,  1.50it/s]


Image 820:
✅ Detected PISTOL with 0.95 confidence at [448.4990234375, 247.88584899902344, 639.9848022460938, 588.1008911132812]

Image 821:

Image 822:

Image 823:


Evaluating: 100%|██████████| 414/414 [05:05<00:00,  1.36it/s]


Image 824:

Image 825:

Image 826:

Image 827:


In [10]:
import os
import torch
import torchmetrics
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from tqdm import tqdm
import numpy as np
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report
from PIL import Image

# SOHAS dataset
class SOHASDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        # Ground truth classes
        self.all_classes = ["knife", "pistol", "smartphone", "billete", "monedero", "tarjeta"]
        self.class_to_idx = {k: v + 1 for v, k in enumerate(self.all_classes)}  # Background is 0

        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".xml"))

        img = Image.open(img_path).convert("RGB")
        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.all_classes:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[label])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        if self.transforms:
            img = self.transforms(img)
        else:
            img = ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# Load faster rcnn model
def get_model(num_classes, checkpoint_path=None, device='cuda'):
    # Load pre-trained faster rcnn model
    model = fasterrcnn_resnet50_fpn(pretrained=True)

    # Replace the classifier with a new one (for your dataset)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)


    if checkpoint_path:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    model.to(device)
    return model

# Evaluate the model for weapon detection and classification
def evaluate(model, dataloader, device, idx_to_class):
    # Metrics for weapon detection
    y_true_weapon = []
    y_pred_weapon = []

    # Metrics for classification
    y_true_knife_pistol = []
    y_pred_knife_pistol = []


    correct_weapon = 0
    total_weapon = 0
    correct_knife_pistol = 0
    total_knife_pistol = 0

    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Run the model on the batch of images
            outputs = model(images)

            # Calculate weapon detection
            for target, output in zip(targets, outputs):
                labels = target["labels"]
                pred_labels = output["labels"]
                for true, pred in zip(labels, pred_labels):

                    true_weapon = 1 if idx_to_class[true.item()] in ['knife', 'pistol'] else 0
                    pred_weapon = 1 if idx_to_class[pred.item()] in ['knife', 'pistol'] else 0

                    y_true_weapon.append(true_weapon)
                    y_pred_weapon.append(pred_weapon)

                    # Weapon classification
                    if true_weapon == 1:
                        true_knife_pistol = 1 if idx_to_class[true.item()] == 'knife' else 2
                        pred_knife_pistol = 1 if idx_to_class[pred.item()] == 'knife' else 2
                        y_true_knife_pistol.append(true_knife_pistol)
                        y_pred_knife_pistol.append(pred_knife_pistol)


                    if true_weapon == pred_weapon:
                        correct_weapon += 1
                    total_weapon += 1

                    if true_knife_pistol == pred_knife_pistol:
                        correct_knife_pistol += 1
                    total_knife_pistol += 1

    # Weapon detection evaluation metrics
    precision_weapon = precision_score(y_true_weapon, y_pred_weapon)
    recall_weapon = recall_score(y_true_weapon, y_pred_weapon)
    f1_weapon = f1_score(y_true_weapon, y_pred_weapon)
    accuracy_weapon = accuracy_score(y_true_weapon, y_pred_weapon)

    # Weapon classification evaluation metrics
    precision_knife_pistol = precision_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    recall_knife_pistol = recall_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    f1_knife_pistol = f1_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    accuracy_knife_pistol = accuracy_score(y_true_knife_pistol, y_pred_knife_pistol)

    # Print results
    print("\n[Detection] Weapon vs Non-Weapon:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'No Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {len(y_true_weapon) - sum(y_true_weapon)}")
    print(f"{'Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {sum(y_true_weapon)}")
    print(f"\n    accuracy                           {accuracy_weapon:.2f}       {len(y_true_weapon)}")
    print(f"   macro avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")
    print(f"weighted avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")

    print("\n[Classification] Knife vs Pistol:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'weapon':<20} {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}      {len(y_true_knife_pistol)}")
    print(f"\n    accuracy                           {accuracy_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"   macro avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"weighted avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")


def main():
    # Paths to the test dataset and the pretrained model
    test_img_dir = "/content/drive/MyDrive/SOHAS/test_dataset/images"
    test_ann_dir = "/content/drive/MyDrive/SOHAS/test_dataset/annotations"
    pretrained_model_path = "/content/drive/MyDrive/fasterrcnn_epoch10.pth"

    # GPU setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Dataset and dataloader
    test_dataset = SOHASDataset(test_img_dir, test_ann_dir, transforms=ToTensor())

    print(f"Number of images in the test dataset: {len(test_dataset)}")

    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))

    )
    idx_to_class = {v: k for k, v in test_dataset.class_to_idx.items()}

    # Load the trained model
    model = get_model(num_classes=1 + len(test_dataset.all_classes), checkpoint_path=pretrained_model_path, device=device)

    # Evaluate
    evaluate(model, test_loader, device, idx_to_class)

if __name__ == "__main__":
    main()

Using device: cuda
Number of images in the test dataset: 828


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Evaluating: 100%|██████████| 414/414 [00:21<00:00, 19.53it/s]



[Detection] Weapon vs Non-Weapon:
                     precision    recall  f1-score   support
No Weapon            0.82      1.00      0.90      299
Weapon               0.82      1.00      0.90      518

    accuracy                           0.86       817
   macro avg       0.82      1.00      0.90       817
weighted avg       0.82      1.00      0.90       817

[Classification] Knife vs Pistol:
                     precision    recall  f1-score   support
weapon               0.97      0.95      0.96      518

    accuracy                           0.98       518
   macro avg       0.97      0.95      0.96       518
weighted avg       0.97      0.95      0.96       518


In [19]:
import os
import torch
import torchmetrics
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from tqdm import tqdm
import numpy as np
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report
from PIL import Image

# Define the SOHAS Dataset
class SOHASDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        self.all_classes = ["knife", "pistol", "smartphone", "billete", "monedero", "tarjeta"]
        self.class_to_idx = {k: v + 1 for v, k in enumerate(self.all_classes)}
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".xml"))

        img = Image.open(img_path).convert("RGB")
        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.all_classes:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[label])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        if self.transforms:
            img = self.transforms(img)
        else:
            img = ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# Load and customise the faster rcnn model
def get_model(num_classes, checkpoint_path=None, device='cuda'):
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    if checkpoint_path:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    model.to(device)
    return model

# Evaluate the model for weapon detection and classification
def evaluate(model, dataloader, device, idx_to_class, confidence_threshold=0.85):
    y_true_weapon = []
    y_pred_weapon = []
    y_true_knife_pistol = []
    y_pred_knife_pistol = []

    correct_weapon = 0
    total_weapon = 0
    correct_knife_pistol = 0
    total_knife_pistol = 0

    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            outputs = model(images)

            for target, output in zip(targets, outputs):
                labels = target["labels"]
                pred_labels = output["labels"]
                scores = output["scores"]

                for true, pred, score in zip(labels, pred_labels, scores):
                    if score < confidence_threshold:
                        continue

                    true_weapon = 1 if idx_to_class[true.item()] in ['knife', 'pistol'] else 0
                    pred_weapon = 1 if idx_to_class[pred.item()] in ['knife', 'pistol'] else 0

                    y_true_weapon.append(true_weapon)
                    y_pred_weapon.append(pred_weapon)

                    if true_weapon == 1:
                        true_knife_pistol = 1 if idx_to_class[true.item()] == 'knife' else 2
                        pred_knife_pistol = 1 if idx_to_class[pred.item()] == 'knife' else 2
                        y_true_knife_pistol.append(true_knife_pistol)
                        y_pred_knife_pistol.append(pred_knife_pistol)

                    if true_weapon == pred_weapon:
                        correct_weapon += 1
                    total_weapon += 1

                    if true_knife_pistol == pred_knife_pistol:
                        correct_knife_pistol += 1
                    total_knife_pistol += 1

    # Weapon detection evluation metrics
    precision_weapon = precision_score(y_true_weapon, y_pred_weapon)
    recall_weapon = recall_score(y_true_weapon, y_pred_weapon)
    f1_weapon = f1_score(y_true_weapon, y_pred_weapon)
    accuracy_weapon = accuracy_score(y_true_weapon, y_pred_weapon)

    # Weapon classification evaluation metrics
    precision_knife_pistol = precision_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    recall_knife_pistol = recall_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    f1_knife_pistol = f1_score(y_true_knife_pistol, y_pred_knife_pistol, average='macro')
    accuracy_knife_pistol = accuracy_score(y_true_knife_pistol, y_pred_knife_pistol)

    print("\n[Detection] Weapon vs Non-Weapon:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'No Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {len(y_true_weapon) - sum(y_true_weapon)}")
    print(f"{'Weapon':<20} {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}      {sum(y_true_weapon)}")
    print(f"\n    accuracy                           {accuracy_weapon:.2f}       {len(y_true_weapon)}")
    print(f"   macro avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")
    print(f"weighted avg       {precision_weapon:.2f}      {recall_weapon:.2f}      {f1_weapon:.2f}       {len(y_true_weapon)}")

    print("\n[Classification] Knife vs Pistol:")
    print(f"{'':<20} precision    recall  f1-score   support")
    print(f"{'weapon':<20} {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}      {len(y_true_knife_pistol)}")
    print(f"\n    accuracy                           {accuracy_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"   macro avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")
    print(f"weighted avg       {precision_knife_pistol:.2f}      {recall_knife_pistol:.2f}      {f1_knife_pistol:.2f}       {len(y_true_knife_pistol)}")

# Load the test image and annotations pathways and the pretrained model pathway
def main():
    test_img_dir = "/content/drive/MyDrive/SOHAS/test_dataset/images"
    test_ann_dir = "/content/drive/MyDrive/SOHAS/test_dataset/annotations"
    pretrained_model_path = "/content/drive/MyDrive/fasterrcnn_epoch10.pth"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    test_dataset = SOHASDataset(test_img_dir, test_ann_dir, transforms=ToTensor())
    print(f"Number of images in the test dataset: {len(test_dataset)}")

    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=lambda x: tuple(zip(*x)))
    idx_to_class = {v: k for k, v in test_dataset.class_to_idx.items()}

    model = get_model(num_classes=1 + len(test_dataset.all_classes), checkpoint_path=pretrained_model_path, device=device)

    evaluate(model, test_loader, device, idx_to_class, confidence_threshold=0.85)

if __name__ == "__main__":
    main()


Using device: cuda
Number of images in the test dataset: 828


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Evaluating: 100%|██████████| 414/414 [00:22<00:00, 18.26it/s]


[Detection] Weapon vs Non-Weapon:
                     precision    recall  f1-score   support
No Weapon            0.96      1.00      0.98      88
Weapon               0.96      1.00      0.98      364

    accuracy                           0.96       452
   macro avg       0.96      1.00      0.98       452
weighted avg       0.96      1.00      0.98       452

[Classification] Knife vs Pistol:
                     precision    recall  f1-score   support
weapon               1.00      0.99      0.99      364

    accuracy                           1.00       364
   macro avg       1.00      0.99      0.99       364
weighted avg       1.00      0.99      0.99       364
